# L08 Pytest, Pip, cProfile - Exam Exercises
Yagmur Yesilyurt

## Q2: My own test

I wrote a small module with a few utility functions and tested them with pytest.
The GitHub Action is in `.github/workflows/run_tests.yml` and runs on every push.

In [ ]:
# the module we are testing -- same my_std function from L02

import numpy as np

def my_std(a):
    """compute standard deviation of a 1D array from scratch"""
    a = np.asarray(a, dtype=float)
    if len(a) == 0:
        raise ValueError("input array is empty")
    mean = np.sum(a) / len(a)
    variance = np.sum((a - mean)**2) / len(a)
    return np.sqrt(variance)

def my_mean(a):
    """compute mean of a 1D array"""
    a = np.asarray(a, dtype=float)
    if len(a) == 0:
        raise ValueError("input array is empty")
    return np.sum(a) / len(a)

In [ ]:
# unit tests -- normally these live in a test_mymodule.py file
# running them here just to show they work

import pytest

def test_my_std_basic():
    """std of a known array"""
    data = [2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]
    assert abs(my_std(data) - 2.0) < 1e-10

def test_my_std_matches_numpy():
    """should agree with numpy for random data"""
    rng = np.random.default_rng(0)
    data = rng.normal(size=500)
    assert abs(my_std(data) - np.std(data)) < 1e-12

def test_my_std_single_element():
    """std of one element should be zero"""
    assert my_std([42.0]) == 0.0

def test_my_std_empty_raises():
    """empty array should raise ValueError"""
    with pytest.raises(ValueError):
        my_std([])

def test_my_mean_basic():
    assert my_mean([1, 2, 3, 4, 5]) == 3.0

# run them all here
test_my_std_basic()
test_my_std_matches_numpy()
test_my_std_single_element()
test_my_std_empty_raises()
test_my_mean_basic()
print("all tests passed!")

The GitHub Action in `.github/workflows/run_tests.yml` runs `pytest` on every push.
If any test fails, the commit is marked as broken and merging is blocked.

## Q4: So slow — profiling

In [ ]:
import cProfile
import pstats
import io
from numba import njit

# slow version: pure python loop to compute pairwise distances
def pairwise_distances_slow(points):
    """compute all pairwise distances between N 2D points"""
    n = len(points)
    dists = []
    for i in range(n):
        for j in range(i+1, n):
            dx = points[i][0] - points[j][0]
            dy = points[i][1] - points[j][1]
            dists.append((dx**2 + dy**2)**0.5)
    return dists

In [ ]:
# profile the slow version to find the bottleneck
rng = np.random.default_rng(1)
points = rng.random((400, 2)).tolist()

pr = cProfile.Profile()
pr.enable()
pairwise_distances_slow(points)
pr.disable()

stream = io.StringIO()
ps = pstats.Stats(pr, stream=stream).sort_stats('cumulative')
ps.print_stats(8)
print(stream.getvalue())

In [ ]:
# fast version: numpy broadcasting -- no Python loops at all
def pairwise_distances_fast(points):
    pts = np.asarray(points)
    # broadcasting trick: (N,1,2) - (1,N,2) gives (N,N,2) difference matrix
    diff = pts[:, None, :] - pts[None, :, :]
    dist_matrix = np.sqrt((diff**2).sum(axis=-1))
    # return upper triangle only (same as slow version)
    i, j = np.triu_indices(len(pts), k=1)
    return dist_matrix[i, j]

In [ ]:
import timeit

pts_list  = points
pts_array = np.array(points)

t_slow = timeit.timeit(lambda: pairwise_distances_slow(pts_list),  number=20)
t_fast = timeit.timeit(lambda: pairwise_distances_fast(pts_array), number=20)

print(f"slow (pure Python): {t_slow:.3f} s")
print(f"fast (numpy):       {t_fast:.3f} s")
print(f"speedup: {t_slow/t_fast:.1f}x")

# sanity check: both give the same result
slow_result = np.sort(pairwise_distances_slow(pts_list))
fast_result = np.sort(pairwise_distances_fast(pts_array))
print(f"results match: {np.allclose(slow_result, fast_result)}")

The profiler showed that almost all the time was spent in the inner loop. Replacing it with numpy broadcasting gives a big speedup with much less code.